In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import json
import pandas as pd
from datasets import load_dataset
import chromadb
from rag_bench import evaluator, helper

from pipeline.ingestion.chroma import insert_data
from pipeline.retriever import ChromaRetriever
from pipeline.reranker import CrossEncoderReranker

### Загрузка датасетов

In [4]:
CACHE_DIR = "../datasets/"

HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"
PUBLIC_TEXTS_REPO: str = "ai-forever/hist-rag-bench-public-texts"
PUBLIC_QA_REPO: str = "ai-forever/hist-rag-bench-public-questions"


def get_public_to_private_texts_mapping(version):
    private_texts_ds = load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version, cache_dir=CACHE_DIR)
    return {item["public_id"]: item["id"] for item in private_texts_ds["train"]}


version = helper.get_latest_version(helper.get_ds_versions(PUBLIC_TEXTS_REPO))
texts_ds = load_dataset(PUBLIC_TEXTS_REPO, revision=version, cache_dir=CACHE_DIR)
qa_dataset = load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version, cache_dir=CACHE_DIR)
mapping = get_public_to_private_texts_mapping(version)
version

'1.15.0'

In [5]:
CHROMA_PATH: str = "../chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)


def create_collection(chroma_client, name):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    collection = chroma_client.create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"},
    )
    print(f"Коллекция '{name}' создана")
    return collection

### Индексация (recursive_character, chunk_size=500)

In [6]:
collection = create_collection(chroma_client, name="reranker_bench")
insert_data(
    dataset=texts_ds,
    collection=collection,
    split_type="recursive_character",
    chunk_size=500,
    chunk_overlap=50,
)

Коллекция 'reranker_bench' создана
Добавлено 2354 чанков в коллекцию 'reranker_bench'


2354

### Вспомогательные функции

In [7]:
def evaluate_retriever(collection, qa_dataset, mapping, retrieve_n=3, reranker=None, max_n=5):
    public_questions = [
        {"id": str(row["public_id"]), "question": row["question"]}
        for row in qa_dataset["train"]
    ]
    results = {}
    for q in public_questions:
        q_id = q["id"]
        raw = ChromaRetriever(collection).top_n(query=q["question"], n=retrieve_n)
        documents = raw["documents"][0]
        metadatas = raw["metadatas"][0]

        if reranker is not None:
            documents, metadatas, _ = reranker.rerank(
                query=q["question"],
                documents=documents,
                metadatas=metadatas,
                max_n=max_n,
            )

        found_ids = [int(m["id"]) for m in metadatas]
        results[q_id] = {"found_ids": found_ids, "model_answer": "-"}

    ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
    return ev_res["average_metrics"]["retrieval"]

## Бенчмарк: Baseline vs Cross-Encoder Reranker

- **Baseline**: retrieve top-3 из векторного поиска, без реранкера
- **Reranker**: retrieve top-10, затем cross-encoder отбирает лучшие 3

In [8]:
reranker = CrossEncoderReranker()

print("Оцениваем baseline (top-3)...")
baseline_metrics = evaluate_retriever(collection, qa_dataset, mapping, retrieve_n=3)
print(f"Baseline: {baseline_metrics}")

print("\nОцениваем с реранкером — dynamic top-k (retrieve-10 → rerank → 1..5)...")
reranker_metrics = evaluate_retriever(
    collection, qa_dataset, mapping,
    retrieve_n=10,
    reranker=reranker,
    max_n=5,
)
print(f"Reranker (dynamic): {reranker_metrics}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 618.06it/s, Materializing param=classifier.weight]                                      


Оцениваем baseline (top-3)...
Baseline: {'hit_rate': np.float64(0.8766666666666667), 'mrr': np.float64(0.8219444444444444), 'precision': np.float64(0.5127777777777777)}

Оцениваем с реранкером — dynamic top-k (retrieve-10 → rerank → 1..5)...
Reranker (dynamic): {'hit_rate': np.float64(0.9216666666666666), 'mrr': np.float64(0.8571388888888888), 'precision': np.float64(0.6199722222222223)}


### Результаты

In [9]:
df = pd.DataFrame({
    "baseline (top-3)": baseline_metrics,
    "reranker (top-10→3)": reranker_metrics,
}).T
df.index.name = "pipeline"
df

,hit_rate,mrr,precision
pipeline,,,
baseline (top-3),0.876667,0.821944,0.512778
reranker (top-10→3),0.921667,0.857139,0.619972


In [ ]:
with open("../results/reranker_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump({"baseline": baseline_metrics, "reranker": reranker_metrics}, f, ensure_ascii=False, indent=2)

print("Результаты сохранены: results/reranker_benchmark_results.json")